# 02. Два тренда и проверка разницы наклонов (методы 4 и 5)

**Метод 4** — две отдельные прямые: для США и для Китая отдельно оценивается  
`y = a + b × год`.

**Метод 5** — одна регрессия на двух странах:  
`y = a + b × год + c × (страна=Китай) + d × (год × (страна=Китай))`.

- `b` — наклон США;
- `b + d` — наклон Китая;
- **`d`** — насколько наклон Китая отличается от наклона США.

Если доверительный интервал для `d` не содержит ноль, разница наклонов в **этой** спецификации статистически отличима от нуля. Это описание двух прямых, а не доказательство, что «политика Китая ускорила науку».

В отчёте считаем два ряда: число статей Scopus и GERD в процентах ВВП.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = Path("..").resolve()
PANEL_PATH = ROOT / "data_reviewed" / "core_panel_reviewed.csv"
TABLES = ROOT / "data_reviewed" / "tables_reviewed"
RAW = ROOT / "data" / "raw"
OUT_FIG = ROOT / "notebooks" / "figures"
OUT_TAB = ROOT / "notebooks" / "tables"
OUT_FIG.mkdir(parents=True, exist_ok=True)
OUT_TAB.mkdir(parents=True, exist_ok=True)

panel = pd.read_csv(PANEL_PATH)
panel = panel[(panel["year"] >= 2010) & (panel["year"] <= 2024)].copy()
US_CN = panel[panel["country_iso3"].isin(["USA", "CHN"])].copy()

def cagr(start: float, end: float, n_years: int) -> float:
    """Compound annual growth rate; n_years = end_year - start_year."""
    if start is None or end is None or np.isnan(start) or np.isnan(end) or start <= 0 or n_years <= 0:
        return np.nan
    return (end / start) ** (1 / n_years) - 1

print("panel rows", len(panel), "US+CN rows", len(US_CN))
print("countries", sorted(panel["country_iso3"].unique()))

import statsmodels.formula.api as smf


panel rows 120 US+CN rows 30
countries ['CHN', 'DEU', 'FRA', 'GBR', 'ISR', 'JPN', 'KOR', 'USA']


In [2]:
def fit_separate_slopes(df: pd.DataFrame, ycol: str, year0: int = 2010, year1: int = 2024):
    """Method 4: OLS y ~ year separately for USA and CHN."""
    rows = []
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for iso, color in [("USA", "#1f77b4"), ("CHN", "#d62728")]:
        sub = df[(df["country_iso3"] == iso) & (df["year"] >= year0) & (df["year"] <= year1)][
            ["year", ycol]
        ].dropna()
        sub = sub.rename(columns={ycol: "y"})
        m = smf.ols("y ~ year", data=sub).fit(cov_type="HC1")
        rows.append({
            "series": ycol,
            "country": iso,
            "n": int(m.nobs),
            "intercept": m.params["Intercept"],
            "slope_b": m.params["year"],
            "slope_se": m.bse["year"],
            "slope_p": m.pvalues["year"],
            "r2": m.rsquared,
            "window": f"{year0}-{year1}",
        })
        ax.scatter(sub["year"], sub["y"], color=color, alpha=0.7, label=f"{iso} данные")
        xgrid = np.linspace(sub["year"].min(), sub["year"].max(), 50)
        ax.plot(xgrid, m.params["Intercept"] + m.params["year"] * xgrid, color=color, label=f"{iso} прямая")
    ax.set_title(f"Метод 4: отдельные прямые — {ycol}")
    ax.set_xlabel("Год")
    ax.set_ylabel(ycol)
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT_FIG / f"02_method4_{ycol}.png", dpi=150)
    plt.show()
    return pd.DataFrame(rows)


def fit_interaction(df: pd.DataFrame, ycol: str, year0: int = 2010, year1: int = 2024):
    """Method 5: y ~ year + China + year:China."""
    sub = df[(df["country_iso3"].isin(["USA", "CHN"])) & (df["year"] >= year0) & (df["year"] <= year1)][
        ["country_iso3", "year", ycol]
    ].dropna().copy()
    sub = sub.rename(columns={ycol: "y"})
    sub["China"] = (sub["country_iso3"] == "CHN").astype(int)
    m = smf.ols("y ~ year + China + year:China", data=sub).fit(cov_type="HC1")
    d = m.params["year:China"]
    ci_low, ci_high = m.conf_int().loc["year:China"]
    out = pd.DataFrame([{
        "series": ycol,
        "n": int(m.nobs),
        "b_USA_slope": m.params["year"],
        "c_China_level": m.params["China"],
        "d_slope_diff": d,
        "d_se": m.bse["year:China"],
        "d_p": m.pvalues["year:China"],
        "d_ci_low": ci_low,
        "d_ci_high": ci_high,
        "d_excludes_zero": not (ci_low <= 0 <= ci_high),
        "r2": m.rsquared,
        "window": f"{year0}-{year1}",
    }])
    print(m.summary().tables[1])
    return out, m


## Статьи Scopus


In [3]:
sep_art = fit_separate_slopes(US_CN, "scopus_articles", year0=2010, year1=2024)
int_art, _ = fit_interaction(US_CN, "scopus_articles", year0=2010, year1=2024)
sep_art


                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept  -4.655e+06   1.56e+06     -2.982      0.003   -7.71e+06    -1.6e+06
year        2523.8946    774.937      3.257      0.001    1005.047    4042.743
China      -1.025e+08   1.16e+07     -8.844      0.000   -1.25e+08   -7.98e+07
year:China  5.087e+04   5746.660      8.853      0.000    3.96e+04    6.21e+04


C:\Users\danii\AppData\Local\Temp\ipykernel_8384\1956295742.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,series,country,n,intercept,slope_b,slope_se,slope_p,r2,window
0,scopus_articles,USA,15,-4.654960e+06,2523.894571,774.93669,1.126344e-03,0.538840,2010-2024
1,scopus_articles,CHN,15,-1.071382e+08,53396.712571,5694.17056,6.759732e-21,0.903464,2010-2024


In [4]:
int_art


,series,n,b_USA_slope,c_China_level,d_slope_diff,d_se,d_p,d_ci_low,d_ci_high,d_excludes_zero,r2,window
0,scopus_articles,30,2523.894571,-1.024833e+08,50872.818,5746.660355,8.551266e-19,39609.570672,62136.065328,True,0.913889,2010-2024


## GERD в процентах ВВП


In [5]:
sep_gerd = fit_separate_slopes(US_CN, "gerd_pct_gdp", year0=2010, year1=2023)
int_gerd, _ = fit_interaction(US_CN, "gerd_pct_gdp", year0=2010, year1=2023)
sep_gerd


                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   -142.6040     17.254     -8.265      0.000    -176.421    -108.787
year           0.0722      0.009      8.441      0.000       0.055       0.089
China         18.8725     18.341      1.029      0.303     -17.074      54.819
year:China    -0.0098      0.009     -1.078      0.281      -0.028       0.008


C:\Users\danii\AppData\Local\Temp\ipykernel_8384\1956295742.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,series,country,n,intercept,slope_b,slope_se,slope_p,r2,window
0,gerd_pct_gdp,USA,14,-142.604027,0.072205,0.008554,3.137139e-17,0.860291,2010-2023
1,gerd_pct_gdp,CHN,14,-123.731571,0.062406,0.003086,6.295877e-91,0.962529,2010-2023


In [6]:
int_gerd


,series,n,b_USA_slope,c_China_level,d_slope_diff,d_se,d_p,d_ci_low,d_ci_high,d_excludes_zero,r2,window
0,gerd_pct_gdp,28,0.072205,18.872455,-0.0098,0.009093,0.28118,-0.027623,0.008023,False,0.970991,2010-2023


In [7]:
summary = pd.concat([sep_art, sep_gerd], ignore_index=True)
interaction = pd.concat([int_art, int_gerd], ignore_index=True)
summary.to_csv(OUT_TAB / "02_method4_separate_slopes.csv", index=False)
interaction.to_csv(OUT_TAB / "02_method5_interaction.csv", index=False)
interaction


,series,n,b_USA_slope,c_China_level,d_slope_diff,d_se,d_p,d_ci_low,d_ci_high,d_excludes_zero,r2,window
0,scopus_articles,30,2523.894571,-1.024833e+08,50872.8180,5746.660355,8.551266e-19,39609.570672,62136.065328,True,0.913889,2010-2024
1,gerd_pct_gdp,28,0.072205,1.887246e+01,-0.0098,0.009093,2.811799e-01,-0.027623,0.008023,False,0.970991,2010-2023


## Краткий вывод для отчёта (методы 4–5)

На графике видны две прямые (метод 4). Таблица с `d` (метод 5) показывает, отличима ли разница наклонов. Интерпретация: описание темпов изменения рядов за 2010–2023, без каузального языка.
